### Visualize Table 41 from Jurgens et al. 2024, which shows overlap between their study and Zheng et al. 2024

In [1]:
import pandas as pd
import numpy as np

In [2]:
GWAS_df = pd.read_csv("Tadros_2025_HCM_MTAG_GWAS.csv", header = 0)
GWAS_df = GWAS_df.iloc[1:].reset_index(drop=True)

In [3]:
GWAS_df.head()

,Locus\nNumber,Locus Name*,Most severe consequence (VEP)##,Pubmed ID if previously published,meta_gws,mtag_novel,Lead SNP,GRCh37,EA,NEA,...,Unnamed: 31,straincirc,Unnamed: 33,Unnamed: 34,LVESVi,Unnamed: 36,Unnamed: 37,LVconc,Unnamed: 39,Unnamed: 40
0,1.0,FAAP20/SKI,3_prime_UTR_variant,NaN,NaN,1.0,rs2503715,1:2144107,G,A,...,8.6E-04,-0.206,0.036,5.3E-09,-0.431,0.085,2.0E-07,0.00521,0.00084,3.5E-10
1,2.0,RNF207,intron_variant,NaN,NaN,1.0,rs11121483,1:6263792,A,G,...,8.9E-03,-0.100,0.024,1.4E-05,-0.306,0.057,4.9E-08,0.00191,0.00056,6.8E-04
2,3.0,HSPB7,3_prime_UTR_variant,"33495596, 33495597",1.0,0.0,rs1048302,1:16340879,T,G,...,1.1E-10,-0.207,0.024,6.2E-19,-0.514,0.058,1.4E-19,0.00214,0.00057,9.1E-05
3,4.0,DNAJB4/NEXN,intron_variant,NaN,NaN,1.0,rs6699769,1:78504264,A,G,...,6.6E-03,-0.087,0.028,1.6E-03,-0.145,0.067,2.3E-02,0.00137,0.00066,2.8E-02
4,5.0,KYAT3/CCBL2,intron_variant,NaN,NaN,1.0,rs2810883,1:89332165,C,T,...,6.7E-05,-0.091,0.023,3.2E-04,-0.167,0.055,4.2E-03,0.00185,0.00054,8.7E-04


In [4]:
GWAS_df.columns

Index(['Locus\nNumber', 'Locus Name*', 'Most severe consequence (VEP)##',
       'Pubmed ID if previously published', 'meta_gws', 'mtag_novel',
       'Lead SNP', 'GRCh37', 'EA', 'NEA', 'HCM MTAG', 'Unnamed: 11',
       'Unnamed: 12', 'Unnamed: 13', 'HCM GWAS#', 'Unnamed: 15', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'HCMSARC+', 'Unnamed: 21',
       'Unnamed: 22', 'HCMSARC-', 'Unnamed: 24', 'Unnamed: 25', 'oHCM',
       'Unnamed: 27', 'Unnamed: 28', 'nHCM', 'Unnamed: 30', 'Unnamed: 31',
       'straincirc', 'Unnamed: 33', 'Unnamed: 34', 'LVESVi', 'Unnamed: 36',
       'Unnamed: 37', 'LVconc', 'Unnamed: 39', 'Unnamed: 40'],
      dtype='object')

#### Get the location of these SNPs, which are in GRCh37 and the prioritized gene. Save this as a bed file for liftover

In [5]:
GWAS_bed_df = GWAS_df[["Lead SNP", "GRCh37", "Locus Name*"]]
GWAS_bed_df.head()

,Lead SNP,GRCh37,Locus Name*
0,rs2503715,1:2144107,FAAP20/SKI
1,rs11121483,1:6263792,RNF207
2,rs1048302,1:16340879,HSPB7
3,rs6699769,1:78504264,DNAJB4/NEXN
4,rs2810883,1:89332165,KYAT3/CCBL2


In [6]:
GWAS_bed_df = GWAS_bed_df.rename(columns = {"Lead SNP": "rsID",
                                           "Locus Name*": "candidate_gene"})

In [7]:
GWAS_bed_df['chr'] = GWAS_bed_df['GRCh37'].str.split(":").str[0]
GWAS_bed_df['start'] = GWAS_bed_df['GRCh37'].str.split(":").str[1]
GWAS_bed_df['end'] = GWAS_bed_df['start']
GWAS_bed_df = GWAS_bed_df[["chr", "start", "end", "rsID", "candidate_gene"]].copy()
GWAS_bed_df

,chr,start,end,rsID,candidate_gene
0,1,2144107,2144107,rs2503715,FAAP20/SKI
1,1,6263792,6263792,rs11121483,RNF207
2,1,16340879,16340879,rs1048302,HSPB7
3,1,78504264,78504264,rs6699769,DNAJB4/NEXN
4,1,89332165,89332165,rs2810883,KYAT3/CCBL2
...,...,...,...,...,...
63,18,55922789,55922789,rs6566955,NEDD4L
64,19,46312077,46312077,rs12460541,DMPK/SYMPK
65,21,30530131,30530131,rs62222424,CCT8
66,22,24161717,24161717,rs5760054,VPREB3/SMARCB1


### Reformat properly for `liftover`

In [8]:
GWAS_bed_df['chr'] = "chr" + GWAS_bed_df['chr'].astype(str)

In [9]:
GWAS_bed_df["start"] = GWAS_bed_df["start"].str.replace(",", "").astype(int)
GWAS_bed_df["end"] = GWAS_bed_df["end"].str.replace(",", "").astype(int)

In [10]:
GWAS_bed_df.head()

,chr,start,end,rsID,candidate_gene
0,chr1,2144107,2144107,rs2503715,FAAP20/SKI
1,chr1,6263792,6263792,rs11121483,RNF207
2,chr1,16340879,16340879,rs1048302,HSPB7
3,chr1,78504264,78504264,rs6699769,DNAJB4/NEXN
4,chr1,89332165,89332165,rs2810883,KYAT3/CCBL2


In [11]:
GWAS_bed_df.shape

(68, 5)

### Save to csv, install liftover to liftover from hg19 to hg38

- conda install bioconda::ucsc-liftover

In [12]:
# don't include header, since liftOver doesn't want this
GWAS_bed_df.to_csv("Tadros_GWAS_hits_GRCh37.bed", sep="\t", index=False, header=False)